# Storytelling de Dados com a Biblioteca SDIC
## Bons e Maus Exemplos de Visualização — Emprego na Indústria por Intensidade Tecnológica

Este notebook demonstra, lado a lado, **maus e bons exemplos** de visualização de dados para responder às três perguntas analíticas apresentadas na aula:

| # | Pergunta Analítica |
|---|---|
| 1 | Como o emprego se distribui entre grupos tecnológicos em dois períodos distintos? |
| 2 | O peso relativo de cada grupo mudou significativamente? |
| 3 | Observamos uma transformação estrutural ou apenas crescimento geral? |

> **Como ler este notebook:**  
> ❌ Seções marcadas com ❌ mostram visualizações **problemáticas** e explicam **por quê** não funcionam.  
> ✅ Seções marcadas com ✅ mostram a **solução** com o gráfico mais adequado para a pergunta.

---
## 1. Configuração

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from sdic_libraries.data_access.emprego import Emprego

# Estilo base dos gráficos
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'figure.dpi': 110,
})

COR = {
    'alta':      '#1565C0',
    'med_alta':  '#1976D2',
    'med_baixa': '#F57C00',
    'baixa':     '#E64A19',
}
PALETA = list(COR.values())

emprego = Emprego()
print('Cliente inicializado com sucesso.')

Cliente inicializado com sucesso.


---
## 2. Definição dos Grupos Tecnológicos (CNAE)

Classificação baseada na taxonomia Pavitt / OCDE, usada pelo MDIC para análise de intensidade tecnológica na indústria de transformação.

In [ ]:
GRUPOS_TECNOLOGIA = [
    {
        'nome_grupo': 'Alta tecnologia',
        # Farmacêuticos, eletrônicos, aeroespacial
        'codigos_cnae': ['21', '26', '30']
    },
    {
        'nome_grupo': 'Tecnologia média-alta',
        # Química fina, máquinas, equipamentos elétricos, veículos
        'codigos_cnae': ['20', '27', '28', '29']
    },
    {
        'nome_grupo': 'Tecnologia média-baixa',
        # Petróleo, borracha, metalurgia, produtos de metal, manutenção
        'codigos_cnae': ['19', '22', '23', '24', '25', '33']
    },
    {
        'nome_grupo': 'Baixa tecnologia',
        # Alimentos, têxtil, madeira, papel, móveis
        'codigos_cnae': ['10', '11', '12', '13', '14', '15', '16', '17', '18', '31', '32']
    },
]

# Cores fixas por grupo
COR_GRUPO = {
    'Alta tecnologia':       '#1565C0',
    'Tecnologia média-alta': '#388E3C',
    'Tecnologia média-baixa':'#F57C00',
    'Baixa tecnologia':      '#C62828',
}

---
## 3. Coleta de Dados via SDIC

In [ ]:
# Estoque nacional agrupado por intensidade tecnológica (nível divisão CNAE)
dados = emprego.get_estoque_emprego_nacional_grupos_cnae(
    grupos_cnae=GRUPOS_TECNOLOGIA,
    nivel_cnae=2,
    agregado=True
)

df = pd.DataFrame(dados)
print(f'Registros retornados: {len(df)}')
print(f'Colunas: {list(df.columns)}')
df.head(8)

In [ ]:
# Verificação: total de empregos por ano
df.groupby('ano')['estoque_trabalhadores'].sum().reset_index()

In [ ]:
# Calcular participação percentual por grupo em cada ano
total_ano = df.groupby('ano')['estoque_trabalhadores'].transform('sum')
df['participacao_pct'] = df['estoque_trabalhadores'] / total_ano * 100

# Dois períodos de referência para comparação
ANO_BASE    = df['ano'].min()
ANO_RECENTE = df['ano'].max()

print(f'Período disponível: {ANO_BASE} — {ANO_RECENTE}')

df_base    = df[df['ano'] == ANO_BASE].set_index('nome_grupo')
df_recente = df[df['ano'] == ANO_RECENTE].set_index('nome_grupo')
GRUPOS     = list(COR_GRUPO.keys())

---
# PERGUNTA 1
## Como o emprego se distribui entre grupos tecnológicos em dois períodos distintos?

Queremos entender a **composição** do emprego industrial — não apenas o tamanho total.

### ❌ Mau Exemplo 1.A — Gráfico de pizza para comparar dois períodos

**Problema:** dois gráficos de pizza lado a lado dificultam a comparação entre os períodos.  
O olho humano não é bom para comparar ângulos — e o problema piora com mais de 4 fatias.

Perguntas que o gráfico NÃO responde:
- Qual grupo cresceu mais?
- A proporção mudou ou apenas o total?
- Quanto cada grupo representa, numericamente?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, (ano, dff) in zip(axes, [(ANO_BASE, df_base), (ANO_RECENTE, df_recente)]):
    valores = [dff.loc[g, 'estoque_trabalhadores'] for g in GRUPOS if g in dff.index]
    labels  = [g for g in GRUPOS if g in dff.index]
    ax.pie(valores, labels=labels, autopct='%1.1f%%', startangle=140)
    ax.set_title(f'Ano: {ano}', fontsize=13)

fig.suptitle('Distribuição do Emprego Industrial por Grupo Tecnológico', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('\n⚠️  Por quê este gráfico é ruim?')
print('   • Comparar dois "pizzas" exige que o leitor memorize os ângulos de cada fatia')
print('   • Não é possível identificar rapidamente qual grupo mudou entre os períodos')
print('   • Percentuais dentro das fatias se sobrepõem, prejudicando a leitura')

### ❌ Mau Exemplo 1.B — Tabela numérica sem visualização

**Problema:** a tabela apresenta todos os dados, mas **não comunicam o padrão**.  
O leitor precisa fazer a comparação mentalmente, célula por célula.

In [ ]:
df_tabela = df[df['ano'].isin([ANO_BASE, ANO_RECENTE])][['ano', 'nome_grupo', 'estoque_trabalhadores', 'participacao_pct']]
df_tabela = df_tabela.pivot(index='nome_grupo', columns='ano', values=['estoque_trabalhadores', 'participacao_pct'])
df_tabela.columns = [f'{col[0]}_{col[1]}' for col in df_tabela.columns]
df_tabela = df_tabela.reset_index()

print(df_tabela.to_string(index=False))

print('\n⚠️  Por quê só a tabela é insuficiente?')
print('   • O leitor precisa subtrair manualmente os valores para ver a mudança')
print('   • Padrões e tendências são invisíveis em formato numérico puro')
print('   • Não há hierarquia visual — todos os números parecem igualmente importantes')

### ✅ Bom Exemplo 1 — Barras empilhadas comparando os dois períodos

**Por quê funciona:**
- Barras side-by-side permitem comparação direta entre períodos  
- O empilhamento mostra a composição (100%) sem perder os grupos individuais  
- Rótulos diretamente nas barras eliminam a necessidade de uma legenda separada  
- Cores consistentes permitem rastrear cada grupo entre os dois períodos

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x        = np.array([0, 1.2])          # posições das duas barras
anos     = [ANO_BASE, ANO_RECENTE]
width    = 0.7
bottoms  = np.zeros(2)

for grupo in GRUPOS:
    vals = [
        df_base.loc[grupo, 'participacao_pct']    if grupo in df_base.index    else 0,
        df_recente.loc[grupo, 'participacao_pct'] if grupo in df_recente.index else 0,
    ]
    bars = ax.bar(x, vals, width=width, bottom=bottoms,
                  color=COR_GRUPO[grupo], label=grupo, edgecolor='white', linewidth=0.6)

    # Rótulo dentro da barra se houver espaço
    for xi, val, bot in zip(x, vals, bottoms):
        if val > 5:
            ax.text(xi, bot + val / 2, f'{val:.1f}%',
                    ha='center', va='center', fontsize=8.5,
                    color='white', fontweight='bold')
    bottoms += np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels([str(a) for a in anos], fontsize=13)
ax.set_ylabel('Participação (%)', fontsize=11)
ax.set_ylim(0, 108)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.set_title(
    f'Distribuição do Emprego Formal na Indústria\n'
    f'por Intensidade Tecnológica — {ANO_BASE} vs {ANO_RECENTE}',
    fontsize=12, fontweight='bold'
)

handles = [plt.Rectangle((0,0),1,1, color=COR_GRUPO[g]) for g in GRUPOS]
ax.legend(handles, GRUPOS, loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=2, fontsize=9, frameon=False)

ax.spines['left'].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

print('✅  O leitor consegue responder à pergunta em menos de 5 segundos:')
print('   Qual é o maior grupo? Qual cresceu mais? A composição mudou?')

---
# PERGUNTA 2
## O peso relativo de cada grupo mudou significativamente entre os períodos?

Aqui o interesse é a **dinâmica** — não apenas o estado final, mas a **direção da mudança**.

### ❌ Mau Exemplo 2.A — Linha com valores absolutos

**Problema:** o crescimento total do emprego (mercado em expansão) mascara mudanças na composição.  
Todas as linhas sobem, o que pode levar à conclusão equivocada de que "nada mudou estruturalmente".

In [ ]:
df_series = df.pivot(index='ano', columns='nome_grupo', values='estoque_trabalhadores')

fig, ax = plt.subplots(figsize=(10, 5))

for grupo in GRUPOS:
    if grupo in df_series.columns:
        ax.plot(df_series.index, df_series[grupo] / 1e6,
                label=grupo, color=COR_GRUPO[grupo], linewidth=2.2, marker='o', markersize=4)

ax.set_xlabel('Ano', fontsize=11)
ax.set_ylabel('Trabalhadores (milhões)', fontsize=11)
ax.set_title('Evolução do Estoque de Emprego por Grupo Tecnológico', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, frameon=False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}M'))
plt.tight_layout()
plt.show()

print('\n⚠️  Por quê este gráfico engana?')
print('   • Quando o mercado cresce, TODOS os grupos crescem em termos absolutos')
print('   • Não é possível ver se um grupo ganhou ou perdeu PARTICIPAÇÃO')
print('   • A pergunta é sobre mudança relativa — não crescimento absoluto')

### ❌ Mau Exemplo 2.B — Barras comparando apenas o ano mais recente

**Problema:** sem referência temporal, é impossível saber se a situação atual é resultado de uma mudança ou de algo estável.  
Este gráfico responde "qual o tamanho hoje?" — não "o que mudou?".

In [ ]:
dados_recente = df_recente.loc[GRUPOS, 'estoque_trabalhadores'].values / 1e6

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(GRUPOS, dados_recente,
              color=[COR_GRUPO[g] for g in GRUPOS], edgecolor='white')

for bar, val in zip(bars, dados_recente):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.2f}M', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Trabalhadores (milhões)', fontsize=10)
ax.set_title(f'Estoque de Emprego por Grupo Tecnológico — {ANO_RECENTE}', fontsize=12, fontweight='bold')
ax.set_xticklabels([g.replace(' ', '\n') for g in GRUPOS], fontsize=9)
ax.spines['left'].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

print('\n⚠️  Por quê este gráfico é insuficiente para a pergunta?')
print('   • Mostra o "retrato" de um único ano — sem movimento, sem mudança')
print('   • "Baixa tecnologia é maior" — mas era assim antes? Cresceu mais ou menos que os outros?')
print('   • A pergunta analítica exige comparação temporal')

### ✅ Bom Exemplo 2 — Slope Chart (Gráfico de Inclinação)

**Por quê funciona:**
- Foca na **mudança relativa**, não no tamanho absoluto  
- A inclinação comunica visualmente se o grupo ganhou (↑) ou perdeu (↓) participação  
- Grupos que cruzam as linhas uns dos outros revelam mudança de ranking  
- Anotações nas extremidades eliminam a necessidade de legenda

In [ ]:
df_pct = df.pivot(index='ano', columns='nome_grupo', values='participacao_pct')

fig, ax = plt.subplots(figsize=(9, 6))

x_anos = df_pct.index.tolist()
x_pos  = np.linspace(0, 1, len(x_anos))

for grupo in GRUPOS:
    if grupo not in df_pct.columns:
        continue
    y = df_pct[grupo].values
    cor = COR_GRUPO[grupo]

    ax.plot(x_pos, y, color=cor, linewidth=2, marker='o', markersize=5, zorder=3)

    # Rótulo à esquerda (ano base)
    ax.text(-0.04, y[0], f'{grupo}\n{y[0]:.1f}%',
            ha='right', va='center', fontsize=8, color=cor, fontweight='bold')
    # Rótulo à direita (ano mais recente)
    delta = y[-1] - y[0]
    seta  = '▲' if delta > 0 else '▼'
    ax.text(1.04, y[-1], f'{y[-1]:.1f}%  {seta}{abs(delta):.1f}pp',
            ha='left', va='center', fontsize=8, color=cor, fontweight='bold')

ax.set_xlim(-0.3, 1.45)
ax.set_xticks(x_pos)
ax.set_xticklabels([str(a) for a in x_anos], fontsize=10)
ax.set_ylabel('Participação no emprego formal (%)', fontsize=10)
ax.set_title(
    'Variação do Peso Relativo dos Grupos Tecnológicos\n'
    'no Emprego Formal da Indústria',
    fontsize=12, fontweight='bold'
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.tick_params(left=False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.show()

print('✅  O leitor identifica imediatamente:')
print('   • Quais grupos ganharam participação (inclinação positiva)')
print('   • Quais perderam (inclinação negativa)')
print('   • A magnitude da mudança (∆pp anotado à direita)')

---
# PERGUNTA 3
## Observamos uma transformação estrutural ou apenas crescimento geral do emprego?

Esta pergunta exige separar dois fenômenos que **podem ocorrer simultaneamente**:  
- O mercado pode crescer e **todos** os grupos crescerem juntos (sem mudança estrutural)  
- O mercado pode crescer e **alguns** grupos ganharem mais que outros (com mudança estrutural)

### ❌ Mau Exemplo 3.A — Valores absolutos empilhados sem percentual

**Problema:** a área crescente das barras confirma que o emprego total aumentou, mas  
é impossível saber se a **composição** mudou — que é exatamente o que a pergunta exige.

In [ ]:
df_pivot_abs = df.pivot(index='ano', columns='nome_grupo', values='estoque_trabalhadores')[GRUPOS]

fig, ax = plt.subplots(figsize=(10, 5))
anos    = df_pivot_abs.index.values
bottoms = np.zeros(len(anos))

for grupo in GRUPOS:
    vals = df_pivot_abs[grupo].values / 1e6
    ax.bar(anos, vals, bottom=bottoms, color=COR_GRUPO[grupo],
           label=grupo, edgecolor='white', linewidth=0.4)
    bottoms += vals

ax.set_ylabel('Trabalhadores (milhões)', fontsize=10)
ax.set_title('Evolução do Emprego Industrial por Grupo Tecnológico', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}M'))
ax.legend(fontsize=8, frameon=False, loc='upper left')
plt.tight_layout()
plt.show()

print('\n⚠️  Por quê este gráfico não responde à pergunta?')
print('   • O crescimento total do emprego faz com que TODAS as barras cresçam')
print('   • Mudanças na composição ficam escondidas sob o crescimento geral')
print('   • A pergunta é: "a PROPORÇÃO mudou?" — não "o total cresceu?"')

### ❌ Mau Exemplo 3.B — Gráfico de linha sem anotação do insight

**Problema:** o gráfico de linhas percentuais é correto, mas a **ausência de contexto**  
faz com que o leitor precise interpretar sozinho — sem saber onde olhar nem o que importa.

In [ ]:
df_pivot_pct = df.pivot(index='ano', columns='nome_grupo', values='participacao_pct')[GRUPOS]

fig, ax = plt.subplots(figsize=(10, 5))

for grupo in GRUPOS:
    ax.plot(df_pivot_pct.index, df_pivot_pct[grupo],
            color=COR_GRUPO[grupo], linewidth=2, marker='o', markersize=3)

ax.set_ylabel('%', fontsize=10)
ax.set_title('Participação dos Grupos no Emprego Industrial (%)', fontsize=12, fontweight='bold')
ax.legend(GRUPOS, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

print('\n⚠️  O que falta neste gráfico?')
print('   • Nenhuma anotação indica qual é o insight principal')
print('   • O leitor precisa descobrir sozinho o que aconteceu')
print('   • Sem contexto — "isso é uma mudança grande ou pequena?"')
print('   • O título descreve o DADO, não a MENSAGEM')

### ✅ Bom Exemplo 3 — Painel duplo: crescimento absoluto + composição relativa

**Por quê funciona:**
- O painel **esquerdo** confirma que o mercado cresceu (contexto)  
- O painel **direito** responde diretamente à pergunta (mudança estrutural)  
- Anotações com setas e texto direcionam o olhar do leitor para o insight principal  
- O título descreve a **conclusão**, não apenas o dado

In [ ]:
df_total = df.groupby('ano')['estoque_trabalhadores'].sum().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(
    'O emprego industrial cresceu — e a composição tecnológica se transformou?',
    fontsize=13, fontweight='bold', y=1.02
)

# ── Painel esquerdo: crescimento absoluto ──────────────────────────────────
ax1.fill_between(df_total['ano'], df_total['estoque_trabalhadores'] / 1e6,
                 alpha=0.25, color='#1565C0')
ax1.plot(df_total['ano'], df_total['estoque_trabalhadores'] / 1e6,
         color='#1565C0', linewidth=2.5)

# Anotação de crescimento
v_ini = df_total.iloc[0]['estoque_trabalhadores'] / 1e6
v_fim = df_total.iloc[-1]['estoque_trabalhadores'] / 1e6
cresc = (v_fim / v_ini - 1) * 100
ax1.annotate(
    f'+{cresc:.0f}%\nno período',
    xy=(df_total.iloc[-1]['ano'], v_fim),
    xytext=(-40, -30), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='#1565C0'),
    fontsize=10, color='#1565C0', fontweight='bold'
)

ax1.set_title('Contexto: o mercado cresceu', fontsize=11, color='#555')
ax1.set_ylabel('Trabalhadores (milhões)', fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}M'))
ax1.spines['left'].set_visible(False)
ax1.tick_params(left=False)
ax1.yaxis.grid(True, linestyle='--', alpha=0.4)

# ── Painel direito: composição relativa ────────────────────────────────────
anos     = df_pivot_pct.index.values
bottoms  = np.zeros(len(anos))

for grupo in GRUPOS:
    vals = df_pivot_pct[grupo].values
    ax2.bar(anos, vals, bottom=bottoms, color=COR_GRUPO[grupo],
            label=grupo, edgecolor='white', linewidth=0.3, width=0.8)
    # rótulo apenas nos anos extremos
    for xi, val, bot in zip([anos[0], anos[-1]], [vals[0], vals[-1]], [bottoms[0], bottoms[-1]]):
        if val > 6:
            ax2.text(xi, bot + val / 2, f'{val:.0f}%',
                     ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
    bottoms += vals

# Identificar grupo com maior variação
variacoes = {g: df_pivot_pct[g].iloc[-1] - df_pivot_pct[g].iloc[0]
             for g in GRUPOS if g in df_pivot_pct.columns}
g_max_var = max(variacoes, key=lambda g: abs(variacoes[g]))

ax2.set_title(
    f'A composição mudou: {g_max_var}\ncom maior variação ({variacoes[g_max_var]:+.1f} pp)',
    fontsize=10, color='#333'
)
ax2.set_ylabel('Participação (%)', fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
           ncol=2, fontsize=8, frameon=False)
ax2.spines['left'].set_visible(False)
ax2.tick_params(left=False)
ax2.yaxis.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print('✅  O painel duplo responde as duas perguntas de uma vez:')
print('   ESQUERDA → "O mercado cresceu? Quanto?" (contexto)')
print('   DIREITA  → "A composição mudou?" (insight principal)')
print()
print('   O título da figura comunica a CONCLUSÃO, não apenas o dado.')

---
## Resumo: Princípios Aplicados

| Pergunta | ❌ Mau exemplo | ✅ Bom exemplo | Princípio |
|---|---|---|---|
| Distribuição em dois períodos | Pizza dupla | Barras empilhadas lado a lado | Comparação direta entre períodos |
| Variação do peso relativo | Linhas com valores absolutos | Slope Chart com ∆pp | Mostrar mudança, não tamanho |
| Transformação estrutural | Empilhado absoluto | Painel duplo absoluto + percentual | Separar contexto de insight |

### Regras práticas

1. **Escolha o gráfico pela pergunta** — não pelo dado disponível  
2. **Normalize quando comparar** — participação percentual revela estrutura; absoluto revela volume  
3. **Anote o insight principal** — o leitor não deve precisar descobrir o que importa  
4. **Títulos descrevem conclusões** — não apenas o dado plotado  
5. **Cores têm memória** — use a mesma cor para o mesmo grupo em todos os gráficos do notebook

---
## Bônus: CAGED — Dinâmica Mensal de Admissões e Desligamentos

O mesmo raciocínio de storytelling se aplica ao CAGED.  
Exemplo de consulta do saldo de movimentações para os grupos tecnológicos.

In [ ]:
# Saldo CAGED nacional — grupos de CNAEs tecnológicos
grupos_caged = [
    {'nome_grupo': 'Alta tecnologia',        'codigos': ['21', '26', '30']},
    {'nome_grupo': 'Tecnologia média-alta',  'codigos': ['20', '27', '28', '29']},
    {'nome_grupo': 'Tecnologia média-baixa', 'codigos': ['19', '22', '23', '24', '25', '33']},
    {'nome_grupo': 'Baixa tecnologia',       'codigos': ['10', '11', '12', '13', '14', '15', '16', '17', '18', '31', '32']},
]

dados_caged = emprego.get_saldo_emprego_detalhado_grupos_cnae(
    grupos_cnae=[
        {'nome_grupo': g['nome_grupo'], 'codigos_cnae': g['codigos']}
        for g in grupos_caged
    ],
    nivel_agregacao='nacional',
    nivel_cnae=2,
)

df_caged = pd.DataFrame(dados_caged)
print(f'Registros: {len(df_caged)}')
print(f'Colunas:   {list(df_caged.columns)}')
df_caged.head()

In [ ]:
# Identificar coluna de data/competência
col_data = next(
    (c for c in df_caged.columns if any(t in c.lower() for t in ['competencia', 'data', 'mes', 'ano'])),
    None
)
col_saldo = next(
    (c for c in df_caged.columns if 'saldo' in c.lower()),
    None
)
col_grupo = next(
    (c for c in df_caged.columns if 'grupo' in c.lower() or 'nome' in c.lower()),
    None
)
print(f'Coluna de data: {col_data}')
print(f'Coluna de saldo: {col_saldo}')
print(f'Coluna de grupo: {col_grupo}')

In [ ]:
if col_data and col_saldo and col_grupo:
    df_caged[col_data] = pd.to_datetime(df_caged[col_data].astype(str), format='%Y%m', errors='coerce')

    # Saldo anual por grupo
    df_caged['ano'] = df_caged[col_data].dt.year
    df_saldo_anual = (
        df_caged.groupby(['ano', col_grupo])[col_saldo]
        .sum()
        .reset_index()
        .rename(columns={col_grupo: 'nome_grupo', col_saldo: 'saldo'})
    )

    # ✅ Bom exemplo: barras divergentes por grupo — saldo positivo/negativo
    df_saldo_pivot = df_saldo_anual.pivot(index='ano', columns='nome_grupo', values='saldo')

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_saldo_pivot))
    n = len(GRUPOS)
    w = 0.18

    for i, grupo in enumerate(GRUPOS):
        if grupo in df_saldo_pivot.columns:
            offset = (i - n / 2 + 0.5) * w
            vals = df_saldo_pivot[grupo].values / 1e3
            colors = [COR_GRUPO[grupo] if v >= 0 else '#BDBDBD' for v in vals]
            ax.bar(x + offset, vals, width=w, color=colors,
                   label=grupo, edgecolor='white', linewidth=0.3)

    ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels(df_saldo_pivot.index.astype(str), fontsize=10)
    ax.set_ylabel('Saldo líquido (mil postos)', fontsize=10)
    ax.set_title(
        'Saldo Anual de Movimentações (CAGED) por Grupo Tecnológico\n'
        'Barras acima de zero = criação de postos; abaixo = destruição',
        fontsize=11, fontweight='bold'
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}k'))
    ax.legend(fontsize=8, frameon=False, loc='upper left')
    ax.spines['left'].set_visible(False)
    ax.tick_params(left=False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Colunas não identificadas automaticamente — ajuste col_data, col_saldo e col_grupo.')